# Seasonal Agriculture Performance Analysis
**AICTE Major Project — VOIS AICTE Batch 1 (2026–2027)**

This notebook analyzes the supplied agricultural dataset to investigate differences in performance across **Kharif, Rabi and Zaid** seasons.

### Analytical questions
- How do yield and profit vary by season?
- Which crops show the strongest average yield?
- How do irrigation methods relate to yield, profit and water efficiency?
- Which variables are associated with yield?
- What unusual or concerning seasonal patterns are visible?
- What evidence-based recommendations can support seasonal planning?

**Interpretation note:** the analysis identifies associations in the supplied dataset; it does not prove causation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("data/seasonal_agriculture.csv")
df = pd.read_csv(DATA_PATH)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Dataset shape:", df.shape)
display(df.head())


## 1. Data quality and preparation

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})
display(quality)
print("Duplicate rows:", df.duplicated().sum())

analysis = df.copy()
for col in analysis.select_dtypes(include=np.number).columns:
    if analysis[col].isna().any():
        analysis[col] = analysis[col].fillna(analysis[col].median())

print("Remaining missing values:", int(analysis.isna().sum().sum()))


## 2. Seasonal performance

In [ ]:
season_summary = (
    analysis.groupby("Season")
    .agg(
        Farms=("Farm_ID", "count"),
        Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha", "mean"),
        Avg_Profit_INR=("Profit_INR", "mean"),
        Total_Profit_INR=("Profit_INR", "sum"),
        Avg_Revenue_INR=("Revenue_INR", "mean"),
        Avg_Cost_INR=("Total_Cost_INR", "mean"),
        Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
        Avg_Disease_Pest_Risk_pct=("Disease_Pest_Risk_pct", "mean"),
    ).reindex(["Kharif", "Rabi", "Zaid"])
)
display(season_summary.round(2))


In [ ]:
season_yield = analysis.groupby("Season")["Yield_Tonnes_Ha"].mean().reindex(["Kharif","Rabi","Zaid"])
plt.figure(figsize=(8,5))
season_yield.plot(kind="bar")
plt.title("Average Yield by Season")
plt.ylabel("Tonnes per Hectare")
plt.xlabel("Season")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
season_profit = analysis.groupby("Season")["Profit_INR"].mean().reindex(["Kharif","Rabi","Zaid"])
plt.figure(figsize=(8,5))
season_profit.plot(kind="bar")
plt.title("Average Profit by Season")
plt.ylabel("INR per Farm")
plt.xlabel("Season")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
positive_profit = (
    analysis.assign(Profitable=analysis["Profit_INR"] > 0)
    .groupby("Season")["Profitable"].mean()
    .reindex(["Kharif","Rabi","Zaid"]) * 100
)
display(positive_profit.round(1).rename("Positive profit rate (%)"))


## 3. Crop-level comparison

In [ ]:
crop_summary = (
    analysis.groupby("Crop")
    .agg(Farms=("Farm_ID","count"),
         Avg_Yield=("Yield_Tonnes_Ha","mean"),
         Avg_Profit=("Profit_INR","mean"),
         Avg_Market_Price=("Market_Price_INR_Tonne","mean"))
    .sort_values("Avg_Yield", ascending=False)
)
display(crop_summary.round(2))

plt.figure(figsize=(10,5))
crop_summary["Avg_Yield"].sort_values().plot(kind="barh")
plt.title("Average Yield by Crop")
plt.xlabel("Tonnes per Hectare")
plt.tight_layout()
plt.show()


## 4. Irrigation analysis

In [ ]:
irrigation_summary = (
    analysis.groupby("Irrigation_Method")
    .agg(Farms=("Farm_ID","count"),
         Avg_Yield=("Yield_Tonnes_Ha","mean"),
         Avg_Profit=("Profit_INR","mean"),
         Avg_Water_Used_m3=("Water_Used_m3","mean"),
         Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean"))
    .sort_values("Avg_Yield", ascending=False)
)
display(irrigation_summary.round(2))


In [ ]:
season_irrigation = (
    analysis.groupby(["Season","Irrigation_Method"])["Yield_Tonnes_Ha"]
    .mean().unstack().reindex(["Kharif","Rabi","Zaid"])
)
display(season_irrigation.round(2))
season_irrigation.plot(kind="bar", figsize=(10,5))
plt.title("Average Yield: Season × Irrigation Method")
plt.ylabel("Tonnes per Hectare")
plt.xlabel("Season")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Relationship analysis

In [ ]:
yield_corr = analysis.select_dtypes(include=np.number).corr()["Yield_Tonnes_Ha"].sort_values(ascending=False)
display(yield_corr.round(3).to_frame("Correlation with Yield_Tonnes_Ha"))

plt.figure(figsize=(8,5))
plt.scatter(analysis["Water_Efficiency_t_per_1000m3"], analysis["Yield_Tonnes_Ha"], alpha=0.35)
plt.title("Yield vs Water Efficiency")
plt.xlabel("Water Efficiency (t per 1,000 m³)")
plt.ylabel("Yield (tonnes/ha)")
plt.tight_layout()
plt.show()


## 6. Statistical comparison

In [ ]:
from scipy.stats import f_oneway

groups = [analysis.loc[analysis["Season"] == s, "Yield_Tonnes_Ha"] for s in ["Kharif","Rabi","Zaid"]]
f_stat, p_value = f_oneway(*groups)
print(f"ANOVA F-statistic: {f_stat:.3f}")
print(f"ANOVA p-value: {p_value:.6g}")
print("Conclusion:", "Seasonal mean yields differ significantly at the 5% level." if p_value < 0.05
      else "No statistically significant seasonal mean-yield difference at the 5% level.")


## 7. Findings and recommendations

In [ ]:
best_yield_season = season_summary["Avg_Yield_Tonnes_Ha"].idxmax()
best_profit_season = season_summary["Avg_Profit_INR"].idxmax()
best_crop = crop_summary["Avg_Yield"].idxmax()
best_irrigation = irrigation_summary["Avg_Yield"].idxmax()
best_efficiency_method = irrigation_summary["Avg_Water_Efficiency"].idxmax()

print("KEY FINDINGS")
print(f"• Highest average yield season: {best_yield_season}")
print(f"• Highest average profit season: {best_profit_season}")
print(f"• Highest average-yield crop: {best_crop}")
print(f"• Highest average-yield irrigation method: {best_irrigation}")
print(f"• Highest average water-efficiency method: {best_efficiency_method}")

print("\nRECOMMENDATIONS")
print("• Use season-specific planning rather than a one-size-fits-all strategy.")
print("• Evaluate irrigation practices together with season, crop and local water availability.")
print("• Investigate the weaker Zaid profitability before expansion decisions.")
print("• Compare crop × irrigation combinations within districts/regions.")
print("• Validate analytical recommendations with field expertise and additional multi-year data.")


## Conclusion
The supplied data shows meaningful differences across seasons, crops and irrigation methods. Kharif performs strongest on average for yield and profit in this sample, while Zaid shows weaker profitability. Irrigation method is associated with differences in yield and water efficiency. These findings are useful for data-driven planning, but they should be treated as descriptive evidence rather than causal conclusions.
